# 01 — Data description, EDA, missing values and baseline
**Competition:** [House Prices – Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)
**Author of this notebook:** Kofi … (team: Akash …, Kofi …, Omisore …)

**Task.** Predict the sale price (`SalePrice`, US dollars) of houses in Ames, Iowa from 79 features.
**Metric.** RMSE between log(predicted price) and log(actual price): errors are measured *relative* to the price,
so the same dollar error costs more on a cheap house than on an expensive one.

> Boxes marked **✍ Kofi** are for your own observations after running the cells above them.
> Write 1–3 sentences in your own words about what you see in the output.

## 0. Setup and data
In Colab: run the next cell and upload `train.csv` and `test.csv` (from the competition's *Data* tab).

In [ ]:
import os
from pathlib import Path
if not (Path("train.csv").exists() and Path("test.csv").exists()):
    try:
        from google.colab import files
        files.upload()          # choose train.csv and test.csv
    except ImportError:
        print("Put train.csv and test.csv next to this notebook.")
print(sorted(p.name for p in Path(".").glob("*.csv")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
SEED = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save(fig, name):
    fig.tight_layout(); fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## 1. Data description

In [ ]:
print(f"train: {train.shape[0]} rows x {train.shape[1]} columns")
print(f"test:  {test.shape[0]} rows x {test.shape[1]} columns")
print("only in train:", sorted(set(train.columns) - set(test.columns)))
num_cols = [c for c in train.select_dtypes(include=np.number).columns if c not in ("Id", "SalePrice")]
cat_cols = [c for c in train.columns if c not in num_cols + ["Id", "SalePrice"]]
print(f"numeric features: {len(num_cols)} | categorical features: {len(cat_cols)}")
train.head()

The most important features (full list: `data_description.txt` on the competition page):

In [ ]:
meaning = {
    "OverallQual": ("ordinal 1–10", "overall material and finish quality"),
    "GrLivArea": ("numeric, sq ft", "above-ground living area"),
    "TotalBsmtSF": ("numeric, sq ft", "total basement area"),
    "1stFlrSF": ("numeric, sq ft", "first floor area"),
    "GarageCars": ("discrete", "garage size in car capacity"),
    "GarageArea": ("numeric, sq ft", "garage area"),
    "YearBuilt": ("numeric, year", "original construction year"),
    "YearRemodAdd": ("numeric, year", "remodel year (= YearBuilt if never remodelled)"),
    "FullBath": ("discrete", "full bathrooms above ground"),
    "TotRmsAbvGrd": ("discrete", "rooms above ground (no bathrooms)"),
    "LotArea": ("numeric, sq ft", "lot size"),
    "Neighborhood": ("categorical, 25 levels", "location within Ames"),
    "MSZoning": ("categorical", "zoning class (residential density, commercial, ...)"),
    "KitchenQual": ("ordinal Po–Ex", "kitchen quality"),
    "ExterQual": ("ordinal Po–Ex", "exterior material quality"),
    "SaleCondition": ("categorical", "normal sale, foreclosure, family sale, partial (new house), ..."),
    "SalePrice": ("numeric, USD", "TARGET: sale price"),
}
pd.DataFrame(meaning, index=["type", "meaning"]).T.assign(
    dtype=lambda d: [str(train[c].dtype) for c in d.index],
    n_unique=lambda d: [train[c].nunique() for c in d.index],
    missing=lambda d: [int(train[c].isna().sum()) for c in d.index])

**✍ Kofi:** How many houses and features are there? How many numeric vs categorical? Is the test set similar in size?

*…your text…*

## 2. Target distribution

In [ ]:
y = train["SalePrice"]
print(y.describe().round(0))
print(f"skewness: raw {y.skew():.2f} | log {np.log1p(y).skew():.2f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(y, bins=60, color="#4C72B0"); ax[0].set(title="SalePrice", xlabel="price, USD", ylabel="number of houses")
ax[1].hist(np.log1p(y), bins=60, color="#DD8452"); ax[1].set(title="log(1 + SalePrice)", xlabel="log price", ylabel="number of houses")
save(fig, "01_target"); plt.show()

**✍ Kofi:** Is the price symmetric or skewed? What are the median and maximum? What does the log transform change,
and why does it fit the competition metric?

*…your text…*

## 3. Distributions of the main features
### 3.1 Numeric features (histograms)

In [ ]:
main_num = ["GrLivArea", "TotalBsmtSF", "LotArea", "YearBuilt", "GarageArea", "1stFlrSF"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), main_num):
    ax.hist(train[c].dropna(), bins=50, color="#55A868")
    ax.set(title=c, xlabel=meaning.get(c, ("", c))[1], ylabel="number of houses")
save(fig, "01_numeric_features"); plt.show()
train[main_num].describe().T.round(1)

**✍ Kofi:** Which features are skewed (long right tail)? Any suspicious values (e.g. very large GrLivArea or LotArea,
basements of size 0)? In which decades were most houses built?

*…your text…*

### 3.2 Categorical features (bar charts)

In [ ]:
main_cat = ["OverallQual", "GarageCars", "KitchenQual", "MSZoning", "BldgType", "SaleCondition"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), main_cat):
    vc = train[c].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color="#8172B2")
    ax.set(title=c, xlabel=c, ylabel="number of houses")
    ax.tick_params(axis="x", rotation=45)
save(fig, "01_categorical_features"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
vc = train["Neighborhood"].value_counts()
ax.bar(vc.index, vc.values, color="#8172B2")
ax.set(title="houses per Neighborhood", xlabel="Neighborhood", ylabel="number of houses")
ax.tick_params(axis="x", rotation=60)
save(fig, "01_neighborhood_counts"); plt.show()

**✍ Kofi:** What is the typical quality level? Which zoning / building type / sale condition dominates?
Are some categories very rare (which could be a problem for the models)?

*…your text…*

## 4. Relations between the features and the target

In [ ]:
logp = np.log1p(train["SalePrice"])
corr = train[num_cols].corrwith(logp).sort_values(ascending=False)
print("correlation with log(SalePrice), top 10:"); print(corr.head(10).round(3))
print("\nmost negative:"); print(corr.tail(5).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(train["GrLivArea"], train["SalePrice"], s=8, alpha=0.5)
out = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
axes[0].scatter(train.loc[out, "GrLivArea"], train.loc[out, "SalePrice"], s=40, color="red", label="outliers")
axes[0].set(title="price vs living area", xlabel="GrLivArea, sq ft", ylabel="SalePrice, USD"); axes[0].legend()
sns.boxplot(x=train["OverallQual"], y=train["SalePrice"], ax=axes[1], color="#4C72B0")
axes[1].set(title="price by overall quality", xlabel="OverallQual", ylabel="SalePrice, USD")
axes[2].scatter(train["YearBuilt"], train["SalePrice"], s=8, alpha=0.5, color="#55A868")
axes[2].set(title="price vs year built", xlabel="YearBuilt", ylabel="SalePrice, USD")
save(fig, "01_relations"); plt.show()

In [ ]:
med = train.groupby("Neighborhood")["SalePrice"].median().sort_values()
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(med.index, med.values / 1000, color="#C44E52")
ax.set(title="median price by Neighborhood", xlabel="Neighborhood", ylabel="median price, thousand USD")
ax.tick_params(axis="x", rotation=60)
save(fig, "01_price_by_neighborhood"); plt.show()

**✍ Kofi:** Which features are most related to price? How does price grow with quality (linear or faster)?
Which neighborhoods are cheapest / most expensive, and by how much do they differ? What is special about the red points?

*…your text…*

## 5. Missing values

In [ ]:
miss = pd.DataFrame({
    "missing_train": train.isna().sum(),
    "missing_test": test.isna().sum().reindex(train.columns),
}).fillna(0).astype(int)
miss["share_train_%"] = (100 * miss["missing_train"] / len(train)).round(1)
miss = miss[(miss.missing_train > 0) | (miss.missing_test > 0)].sort_values("missing_train", ascending=False)
print(f"{len(miss)} columns have missing values")
miss

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
top = miss.head(20)
ax.bar(top.index, top["share_train_%"], color="#DD8452")
ax.set(title="share of missing values in train (top 20 columns)", xlabel="column", ylabel="% of houses")
ax.tick_params(axis="x", rotation=60)
save(fig, "01_missing_values"); plt.show()

Two kinds of missing values (see `data_description.txt`):
* **"NA means the feature does not exist"**: `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`, garage and
  basement columns. Handling: fill with `"None"` (categorical) or `0` (areas, counts). Dropping these houses would
  throw away most of the data, and "no pool" is real information.
* **Truly unknown values**: `LotFrontage` (~18%), a few in `MasVnrType/MasVnrArea`, `Electrical`, and some test
  columns. Handling: `LotFrontage` = median of the same neighborhood (similar streets), the rest = most frequent value.

**✍ Kofi:** Which columns are almost entirely missing? Does the test set have missing values that the train set does not?
Do you agree with the handling above? Why not simply drop rows with missing values?

*…your text…*

## 6. Outliers and other data problems

In [ ]:
print("houses with GrLivArea > 4000 sq ft:")
print(train.loc[train["GrLivArea"] > 4000, ["Id", "GrLivArea", "SalePrice", "OverallQual", "SaleCondition"]])
print("\nUtilities values in train:", train["Utilities"].value_counts().to_dict())

**✍ Kofi:** Describe the two houses that are huge but cheap (look at `SaleCondition`). Why is it reasonable to remove
them from training (as Akash's notebook does), but not the two large *expensive* houses? Which column is almost constant?

*…your text…*

## 7. Baseline (trivial predictions)
Same validation as in Akash's notebook: 5-fold cross-validation (`KFold(5, shuffle=True, random_state=42)`) and
RMSE on log prices. Every "model" below is fitted only on the training folds.

In [ ]:
kf = KFold(5, shuffle=True, random_state=SEED)
def cv_rmse(fit_predict):
    scores = []
    for tr_idx, va_idx in kf.split(train):
        tr, va = train.iloc[tr_idx], train.iloc[va_idx]
        pred = fit_predict(tr, va)
        scores.append(np.sqrt(np.mean((pred - np.log1p(va["SalePrice"])) ** 2)))
    return np.mean(scores), np.std(scores)

def constant_mean(tr, va):          # best constant for RMSE on the log scale
    return np.full(len(va), np.log1p(tr["SalePrice"]).mean())

def median_by_quality(tr, va):      # simple rule: median log price of houses with the same OverallQual
    m = np.log1p(tr["SalePrice"]).groupby(tr["OverallQual"]).median()
    return va["OverallQual"].map(m).fillna(np.log1p(tr["SalePrice"]).median()).to_numpy()

def median_by_neighborhood(tr, va):
    m = np.log1p(tr["SalePrice"]).groupby(tr["Neighborhood"]).median()
    return va["Neighborhood"].map(m).fillna(np.log1p(tr["SalePrice"]).median()).to_numpy()

baselines = pd.DataFrame(
    [(name, *cv_rmse(f)) for name, f in [("constant (mean log price)", constant_mean),
                                         ("rule: median by OverallQual", median_by_quality),
                                         ("rule: median by Neighborhood", median_by_neighborhood)]],
    columns=["baseline", "CV RMSE (log)", "std"]).round(4)
baselines.to_csv("baselines.csv", index=False)
baselines

**✍ Kofi:** How much better than the constant are the simple rules? Compare with Akash's models (CV RMSE ≈ 0.11):
how much of the improvement comes from real modelling?

*…your text…*

## 8. Summary of the EDA
**✍ Kofi:** 4–6 bullet points with your main findings (target, strongest features, missing values, outliers, baseline).

* …